# EXP-0 — the fp16 correctness gate

**Question:** does Qwen2-VL-2B (trained in bf16) survive fp16 on this T4?

**Pass:** a coherent caption of a noise image AND `prompt_tokens` ≈ 284 (256 image tokens + 2 delimiters + ~26 of chat template & question).
**Fail:** `!!!!` gibberish, empty output, or a CUDA device-side assert in the log → we switch to the Phi-3.5-vision fallback.

Run cells top to bottom. Runtime → Change runtime type → **T4 GPU** first.


In [ ]:
# 1. Confirm we actually got a T4
!nvidia-smi -L


In [ ]:
# 2. Install vLLM 0.25.1 built for CUDA 12.9 (5-15 min; installs its own torch).
# The default PyPI wheel targets CUDA 13, which Colab's driver doesn't have --
# it fails with: ImportError: libcudart.so.13. The cu129 wheel matches Colab.
%pip -q install https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1+cu129-cp38-abi3-manylinux_2_28_x86_64.whl --extra-index-url https://download.pytorch.org/whl/cu129
import vllm, transformers
print('vllm', vllm.__version__, '| transformers', transformers.__version__)


## Launch the server

Flags, and why each exists:
- `--dtype half` — fp16; the T4 has no bf16 support (compute capability 7.5 < 8.0). This is the risk EXP-0 tests.
- `--max-model-len 4096` — cap context so the KV cache fits 16 GB.
- `--gpu-memory-utilization 0.85` — leave headroom; vLLM pre-allocates.
- `--enforce-eager` — no CUDA graphs; vision encoders take dynamic shapes and capture can fail (also saves memory).
- `--limit-mm-per-prompt` / `--mm-processor-kwargs max_pixels` — bound the worst-case image so startup profiling doesn't OOM.


In [ ]:
import subprocess

args = [
    'vllm', 'serve', 'Qwen/Qwen2-VL-2B-Instruct',
    '--dtype', 'half',
    '--max-model-len', '4096',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--limit-mm-per-prompt', '{"image": 1}',
    '--mm-processor-kwargs', '{"min_pixels": 784, "max_pixels": 1003520}',
    '--port', '8000',
]
log_file = open('/content/vllm.log', 'w')
server = subprocess.Popen(args, stdout=log_file, stderr=subprocess.STDOUT)
print('server starting, pid', server.pid, '- takes ~2-4 min (downloads 4.4 GB of weights first time)')


In [ ]:
# 3. Wait until the server answers (or die with the log tail)
import time, requests

for attempt in range(120):
    if server.poll() is not None:
        print(open('/content/vllm.log').read()[-3000:])
        raise RuntimeError('server process exited - see log above')
    try:
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            print(f'server ready after ~{attempt*5}s')
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise RuntimeError('server not ready after 10 min - check /content/vllm.log')


## The benchmark code

Identical to `bench/` in the repo (inlined until the repo is on GitHub): calibrated noise image → data URL → streaming request with the stopwatch → gate verdict.


In [ ]:
import base64, io, json, time
import numpy as np
import requests
from PIL import Image

MODEL = 'Qwen/Qwen2-VL-2B-Instruct'


def image_tokens(width, height):
    """Tokens an image costs in Qwen2-VL: one per 28x28 pixel block."""
    return (height // 28) * (width // 28)


def noise_image(side_px, seed):
    """Square RGB random noise; unique per seed so caching can't warm it."""
    if side_px % 28 != 0:
        raise ValueError(f'side_px must be a multiple of 28, got {side_px}')
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (side_px, side_px, 3), dtype=np.uint8))


def to_data_url(img):
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')


def image_payload(model, data_url, question, max_tokens=64):
    return {
        'model': model,
        'messages': [{'role': 'user', 'content': [
            {'type': 'text', 'text': question},
            {'type': 'image_url', 'image_url': {'url': data_url}},
        ]}],
        'temperature': 0,
        'max_tokens': max_tokens,
    }


def send_request(payload, base_url='http://localhost:8000'):
    """The stopwatch: TTFT stops at the first piece of answer text;
    the final SSE chunk carries usage.prompt_tokens (image tokens included)."""
    payload = dict(payload, stream=True, stream_options={'include_usage': True})
    t_start = time.perf_counter()
    ttft, parts, usage = None, [], None
    with requests.post(f'{base_url}/v1/chat/completions', json=payload, stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line.startswith(b'data: '):
                continue
            data = line[len(b'data: '):]
            if data == b'[DONE]':
                break
            chunk = json.loads(data)
            if chunk.get('usage'):
                usage = chunk['usage']
            for choice in chunk.get('choices', []):
                content = choice.get('delta', {}).get('content')
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - t_start
                    parts.append(content)
    return {
        'ttft_s': ttft,
        'e2e_s': time.perf_counter() - t_start,
        'prompt_tokens': usage['prompt_tokens'] if usage else None,
        'completion_tokens': usage['completion_tokens'] if usage else None,
        'output_text': ''.join(parts),
    }


def looks_coherent(text):
    stripped = text.strip()
    return len(stripped) >= 10 and '!!!!' not in stripped and len(set(stripped)) > 5

print('benchmark code loaded')


In [ ]:
# 4. RUN THE GATE
side = 448
payload = image_payload(MODEL, to_data_url(noise_image(side, seed=0)), 'Describe this image.')
result = send_request(payload)

vision_tokens = image_tokens(side, side) + 2   # 256 + start/end delimiters
extra = (result['prompt_tokens'] or 0) - vision_tokens
coherent = looks_coherent(result['output_text'])
formula_ok = 0 < extra <= 40

print('caption:', repr(result['output_text']))
print()
print(f"prompt_tokens={result['prompt_tokens']}  (expected {vision_tokens} vision + ~26 template)")
print(f"ttft={result['ttft_s']:.3f}s  e2e={result['e2e_s']:.3f}s")
print()
print('coherent output:', coherent)
print('token formula holds:', formula_ok)
print()
print('GATE PASSED - fp16 Qwen2-VL works on this T4' if coherent and formula_ok
      else 'GATE FAILED - fall back to Phi-3.5-vision (tell Claude)')


In [ ]:
# 5. What the server logged: vision backend + any errors
!grep -iE 'xformers|vision|backend|error|assert|nan' /content/vllm.log | tail -25


## Done

Copy the output of cells 4 and 5 back into the Claude session. If the gate passed, session 2 runs the real curves (EXP-T and EXP-V) on this same server setup.
